In [2]:
import xarray as xr
import numpy as np
import pandas as pd

DATA_PATH = "../data/west_bengal_climate_final.nc"

ds = xr.open_dataset(DATA_PATH)

variables = ["rainfall", "tmax", "tmin"]

In [3]:
print(ds.data_vars)

Data variables:
    rainfall            (time, latitude, longitude) float64 1MB ...
    tmax                (time, latitude, longitude) float32 563kB ...
    tmin                (time, latitude, longitude) float32 563kB ...
    overlap_percentage  (latitude, longitude) float64 280B ...
    west_bengal_mask    (latitude, longitude) bool 35B ...


In [4]:
data_arrays = []

feature_names = []

for lat in ds.latitude.values:
    for lon in ds.longitude.values:

        # Skip cells that are not part of the selected region
        if "west_bengal_mask" in ds:
            if not bool(ds.west_bengal_mask.sel(latitude=lat, longitude=lon)):
                continue

        for var in variables:

            values = ds[var].sel(
                latitude=lat,
                longitude=lon
            ).values

            data_arrays.append(values)

            feature_names.append(
                f"{var}_lat{lat}_lon{lon}"
            )

In [5]:
data_matrix = np.stack(data_arrays, axis=1)

print(data_matrix.shape)

(4018, 42)


In [6]:
daily_df = pd.DataFrame(
    data_matrix,
    index=pd.to_datetime(ds.time.values),
    columns=feature_names
)

daily_df.head()

,rainfall_lat22.5_lon86.5,tmax_lat22.5_lon86.5,tmin_lat22.5_lon86.5,rainfall_lat22.5_lon87.5,tmax_lat22.5_lon87.5,tmin_lat22.5_lon87.5,rainfall_lat22.5_lon88.5,tmax_lat22.5_lon88.5,tmin_lat22.5_lon88.5,rainfall_lat23.5_lon85.5,...,tmin_lat25.5_lon88.5,rainfall_lat26.5_lon87.5,tmax_lat26.5_lon87.5,tmin_lat26.5_lon87.5,rainfall_lat26.5_lon89.5,tmax_lat26.5_lon89.5,tmin_lat26.5_lon89.5,rainfall_lat27.5_lon88.5,tmax_lat27.5_lon88.5,tmin_lat27.5_lon88.5
2015-01-01,0.000000,23.892088,13.483067,0.000000,23.817703,14.801130,0.110534,24.163929,14.857316,0.000000,...,10.745741,0.000000,24.186161,8.732943,0.000000,25.144722,9.689601,0.000000,24.181528,8.665056
2015-01-02,6.299613,23.295881,16.795536,3.372333,23.607836,17.778358,0.821536,24.505510,18.066349,1.430627,...,13.753613,0.000000,23.935799,10.750012,0.000000,24.642221,11.564000,0.000000,23.787605,10.487411
2015-01-03,12.924912,26.290567,18.714966,1.741906,26.167974,19.789888,0.000000,25.752417,19.968304,10.937878,...,14.941014,7.823306,19.658209,11.926073,0.861086,20.345957,12.969721,0.666592,19.748722,11.653724
2015-01-04,0.134959,26.083162,17.897100,7.189056,26.039606,18.971685,23.144264,26.115292,19.191807,3.023484,...,14.234319,0.158079,21.925797,12.217904,14.990569,22.521511,12.206127,2.804070,22.002872,11.905456
2015-01-05,0.000000,24.977316,14.732183,0.000000,24.779966,15.768693,0.000000,24.622873,16.400702,0.000000,...,13.302844,0.000000,18.527346,11.960801,0.000000,19.886480,11.875532,0.000000,18.868759,11.572913


In [7]:
print("Total missing values:", daily_df.isna().sum().sum())

Total missing values: 0


In [8]:
LOOKBACK = 7
X = []
y = []
prediction_dates = []

values = daily_df.values

for i in range(LOOKBACK, len(values)):

    # Previous 7 days
    X.append(values[i-LOOKBACK:i])

    # Current day = target
    y.append(values[i])

    # Date being predicted
    prediction_dates.append(daily_df.index[i])

X = np.array(X)
y = np.array(y)
prediction_dates = np.array(prediction_dates)